In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(r'C:\Tugas Alprog\files\Kelas B_CO2 Emissions by Car.csv', sep=';')
df.columns = df.columns.str.strip()
df['ENGINE_SIZE'] = pd.to_numeric(df['ENGINE_SIZE'], errors='coerce')
df['CYLINDERS'] = pd.to_numeric(df['CYLINDERS'], errors='coerce')
df['FUEL_CONSUMPTION*'] = pd.to_numeric(df['FUEL_CONSUMPTION*'], errors='coerce')
df['CO2_EMISSIONS'] = pd.to_numeric(df['CO2_EMISSIONS'], errors='coerce')

print("Data loaded:", df.shape)
print(df.head(3))

# ============================================================
# SOAL A - KATEGORI A (AGREGASI)
# Hitung total variasi model (MODEL) unik yang dimiliki tiap MAKE.
# Tampilkan 10 merek paling variatif dalam Bar Chart
# ============================================================
fig_a, ax_a = plt.subplots(figsize=(12, 6))

model_variety = df.groupby('MAKE')['MODEL'].nunique().sort_values(ascending=False).head(10)

colors_a = plt.cm.viridis(np.linspace(0.2, 0.9, len(model_variety)))
bars = ax_a.bar(model_variety.index, model_variety.values, color=colors_a, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, model_variety.values):
    ax_a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
              str(val), ha='center', va='bottom', fontsize=10, fontweight='bold', color='#333333')

ax_a.set_title('Top 10 Merek Mobil dengan Variasi Model Terbanyak\n(Kategori A - Agregasi | Kelompok 4)', 
               fontsize=14, fontweight='bold', pad=15)
ax_a.set_xlabel('Merek Mobil (MAKE)', fontsize=12)
ax_a.set_ylabel('Jumlah Variasi Model Unik', fontsize=12)
ax_a.tick_params(axis='x', rotation=30)
ax_a.grid(axis='y', alpha=0.3, linestyle='--')
ax_a.set_facecolor('#f9f9f9')
fig_a.patch.set_facecolor('white')
plt.tight_layout()
fig_a.savefig(r'C:\Tugas Alprog\files\A_variasi_model_per_make.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Soal A selesai")

# ============================================================
# SOAL B - KATEGORI B (TREN/FILTER)
# Cari mobil tipe COMPACT yang memiliki ENGINE_SIZE > 2.5 liter
# namun emisinya di bawah median emisi mobil compact global.
# Tampilkan modelnya dalam Bar Chart
# ============================================================
compact_df = df[df['VEHICLE CLASS'] == 'COMPACT'].copy()
median_co2_compact = compact_df['CO2_EMISSIONS'].median()
print(f"\nMedian CO2 COMPACT global: {median_co2_compact}")

filtered_compact = compact_df[
    (compact_df['ENGINE_SIZE'] > 2.5) &
    (compact_df['CO2_EMISSIONS'] < median_co2_compact)
].copy()

print(f"Jumlah data compact ENGINE>2.5 & CO2<median: {len(filtered_compact)}")

# Tampilkan per model (ambil rata-rata jika ada duplikat model)
model_co2 = filtered_compact.groupby('MODEL')['CO2_EMISSIONS'].mean().sort_values()

fig_b, ax_b = plt.subplots(figsize=(12, max(5, len(model_co2)*0.4 + 2)))

colors_b = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(model_co2)))
bars_b = ax_b.barh(model_co2.index, model_co2.values, color=colors_b, edgecolor='white')

for bar, val in zip(bars_b, model_co2.values):
    ax_b.text(val + 0.5, bar.get_y() + bar.get_height()/2,
              f'{val:.1f}', va='center', fontsize=9, fontweight='bold')

ax_b.axvline(median_co2_compact, color='red', linestyle='--', linewidth=1.5,
             label=f'Median CO2 Compact = {median_co2_compact:.1f}')
ax_b.set_title(f'Mobil COMPACT: ENGINE_SIZE > 2.5L & CO2 < Median ({median_co2_compact:.1f})\n'
               f'(Kategori B - Filter | Kelompok 4)', fontsize=13, fontweight='bold', pad=15)
ax_b.set_xlabel('CO2 Emissions (g/km)', fontsize=12)
ax_b.set_ylabel('Model Mobil', fontsize=12)
ax_b.legend(fontsize=10)
ax_b.grid(axis='x', alpha=0.3, linestyle='--')
ax_b.set_facecolor('#f9f9f9')
fig_b.patch.set_facecolor('white')
plt.tight_layout()
fig_b.savefig(r'C:\Tugas Alprog\files\B_analisis_median_compact.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Soal B selesai")

# ============================================================
# SOAL C - KATEGORI C (KORELASI)
# Bandingkan hubungan korelasi antara konsumsi bahan bakar
# (FUEL_CONSUMPTION*) dengan emisi karbon (CO2_EMISSIONS).
# Tampilkan dalam Scatter Plot
# ============================================================
fig_c, ax_c = plt.subplots(figsize=(10, 7))

scatter_data = df[['FUEL_CONSUMPTION*', 'CO2_EMISSIONS', 'FUEL']].dropna()

# Warnai berdasarkan jenis bahan bakar
fuel_types = scatter_data['FUEL'].unique()
color_map = {f: plt.cm.tab10(i/len(fuel_types)) for i, f in enumerate(fuel_types)}
colors_c = scatter_data['FUEL'].map(color_map)

sc = ax_c.scatter(scatter_data['FUEL_CONSUMPTION*'], scatter_data['CO2_EMISSIONS'],
                  c=colors_c, alpha=0.5, s=20, edgecolors='none')

# Garis regresi
slope, intercept, r_value, p_value, std_err = stats.linregress(
    scatter_data['FUEL_CONSUMPTION*'], scatter_data['CO2_EMISSIONS'])
x_line = np.linspace(scatter_data['FUEL_CONSUMPTION*'].min(), scatter_data['FUEL_CONSUMPTION*'].max(), 100)
y_line = slope * x_line + intercept
ax_c.plot(x_line, y_line, color='red', linewidth=2, label=f'Regresi Linear\nr = {r_value:.4f}, R² = {r_value**2:.4f}')

# Legend bahan bakar
legend_patches = [mpatches.Patch(color=color_map[f], label=f'Fuel: {f}') for f in fuel_types]
legend_patches.append(plt.Line2D([0], [0], color='red', linewidth=2, label=f'Regresi: r={r_value:.4f}'))
ax_c.legend(handles=legend_patches, fontsize=9, loc='upper left')

ax_c.set_title(f'Korelasi FUEL_CONSUMPTION* vs CO2_EMISSIONS\n'
               f'Pearson r = {r_value:.4f} | R² = {r_value**2:.4f} | p-value = {p_value:.2e}\n'
               f'(Kategori C - Korelasi | Kelompok 4)',
               fontsize=12, fontweight='bold', pad=12)
ax_c.set_xlabel('Fuel Consumption (L/100km)', fontsize=12)
ax_c.set_ylabel('CO2 Emissions (g/km)', fontsize=12)
ax_c.grid(alpha=0.3, linestyle='--')
ax_c.set_facecolor('#f9f9f9')
fig_c.patch.set_facecolor('white')
plt.tight_layout()
fig_c.savefig(r'C:\Tugas Alprog\files\C_korelasi_fuel_co2.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Soal C selesai | r = {r_value:.4f}")

# ============================================================
# SOAL D - KATEGORI D (DISTRIBUSI)
# Tampilkan perbandingan sebaran kapasitas mesin (ENGINE_SIZE)
# berdasarkan jenis tipe bahan bakar (FUEL). Tampilkan dalam Boxplot
# ============================================================
fig_d, ax_d = plt.subplots(figsize=(12, 7))

fuel_groups = [group['ENGINE_SIZE'].dropna().values 
               for name, group in df.groupby('FUEL')]
fuel_labels = [name for name, _ in df.groupby('FUEL')]
fuel_counts = [len(group) for _, group in df.groupby('FUEL')]

bp = ax_d.boxplot(fuel_groups, labels=fuel_labels, patch_artist=True,
                  medianprops=dict(color='red', linewidth=2),
                  whiskerprops=dict(linewidth=1.5),
                  capprops=dict(linewidth=1.5),
                  flierprops=dict(marker='o', markersize=3, alpha=0.4, markeredgewidth=0))

box_colors = plt.cm.Set2(np.linspace(0, 1, len(fuel_labels)))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

# Tambahkan label count
for i, (label, count, group) in enumerate(zip(fuel_labels, fuel_counts, fuel_groups), 1):
    ax_d.text(i, ax_d.get_ylim()[0] if ax_d.get_ylim()[0] > 0 else min(df['ENGINE_SIZE'].dropna())*0.95,
              f'n={count}', ha='center', va='top', fontsize=9, color='#555')

# Tambahkan anotasi median
for i, group in enumerate(fuel_groups, 1):
    med = np.median(group)
    ax_d.text(i + 0.32, med, f'{med:.1f}', va='center', fontsize=8.5, color='darkred', fontweight='bold')

ax_d.set_title('Sebaran Kapasitas Mesin (ENGINE_SIZE) berdasarkan Jenis Bahan Bakar (FUEL)\n'
               '(Kategori D - Distribusi | Kelompok 4)',
               fontsize=13, fontweight='bold', pad=15)
ax_d.set_xlabel('Jenis Bahan Bakar (FUEL)', fontsize=12)
ax_d.set_ylabel('Engine Size (Liter)', fontsize=12)
ax_d.grid(axis='y', alpha=0.3, linestyle='--')
ax_d.set_facecolor('#f9f9f9')
fig_d.patch.set_facecolor('white')

# Keterangan kode fuel
fuel_note = "Keterangan: X=Regular, Z=Premium, D=Diesel, E=Ethanol, N=Natural Gas"
fig_d.text(0.5, -0.02, fuel_note, ha='center', fontsize=9, color='gray', style='italic')

plt.tight_layout()
fig_d.savefig(r'C:\Tugas Alprog\files\D_boxplot_fuel_consumption.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Soal D selesai")

print("\n🎉 Semua soal Kelompok 4 selesai dikerjakan!")
print("File tersimpan di /mnt/user-data/outputs/")


Data loaded: (679, 10)
   MODEL_YEAR   MAKE  MODEL VEHICLE CLASS  ENGINE_SIZE  CYLINDERS  \
0        2001  ACURA  1.7EL       COMPACT          1.7          4   
1        2001  ACURA  1.7EL       COMPACT          1.7          4   
2        2001  ACURA  3.2CL       COMPACT          3.2          6   

  TRANSMISSION FUEL  FUEL_CONSUMPTION*  CO2_EMISSIONS  
0           A4    X                9.3            191  
1           M5    X                8.9            191  
2          AS5    Z               13.7            265  
✅ Soal A selesai

Median CO2 COMPACT global: 246.0
Jumlah data compact ENGINE>2.5 & CO2<median: 2
✅ Soal B selesai
✅ Soal C selesai | r = 0.9748
✅ Soal D selesai

🎉 Semua soal Kelompok 4 selesai dikerjakan!
File tersimpan di /mnt/user-data/outputs/


C:\Users\danis\AppData\Local\Temp\ipykernel_13048\3285228639.py:149: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax_d.boxplot(fuel_groups, labels=fuel_labels, patch_artist=True,
